
Project 3: Machine Learning for Predicting Trading Signals

 This project aims to apply advanced classification techniques to financial market data, facilitating effective trading decision-making.

1. Feature Engineering with Technical Indicator

In this section, i will compute the Moving Average Convergence Divergence (MACD) and Relative Strength Index (RSI) for each stock. These indicators are crucial for generating buy, sell, and hold signals, which will serve as ground-truth labels for our predictive models

1.1 Moving Average Convergence Divergence (MACD)

In [1]:
import pandas as pd
import numpy as np

# Define file paths
historical_stocks_path = "historical_stocks.csv"
historical_stock_prices_path = "historical_stock_prices.csv"

# Load datasets
historical_stocks = pd.read_csv(historical_stocks_path)
historical_stocks_prices = pd.read_csv(historical_stock_prices_path, low_memory=True)

# Merge the datasets (as done previously)
merged_stocks_df = pd.merge(historical_stocks_prices, historical_stocks, on='ticker', how='left')

# Ensure the DataFrame is sorted by ticker and date for correct technical indicator calculations
merged_stocks_df = merged_stocks_df.sort_values(by=['ticker', 'date']).reset_index(drop=True)

# --- Calculate MACD ---
# Function to calculate EMA (Exponential Moving Average)
def calculate_ema(df, span, column='close'):
    return df.groupby('ticker')[column].transform(lambda x: x.ewm(span=span, adjust=False).mean())

# Calculate 12-period EMA
merged_stocks_df['EMA_12'] = calculate_ema(merged_stocks_df, span=12)
# Calculate 26-period EMA
merged_stocks_df['EMA_26'] = calculate_ema(merged_stocks_df, span=26)
# Calculate MACD Line
merged_stocks_df['MACD'] = merged_stocks_df['EMA_12'] - merged_stocks_df['EMA_26']
# Calculate Signal Line (9-period EMA of MACD)
merged_stocks_df['MACD_Signal'] = calculate_ema(merged_stocks_df, span=9, column='MACD')

# --- Generate MACD Signals ---
# Initialize MACD_Action column
merged_stocks_df['MACD_Action'] = 'Hold'

# Buy Signal: MACD crosses above Signal Line
# This often implies MACD > MACD_Signal, and previous MACD <= previous MACD_Signal
# To avoid look-ahead bias, we'll check today's MACD vs Signal and yesterday's values.
# Shift operations are grouped by ticker to avoid crossing stock boundaries.

# Calculate lagged MACD and MACD_Signal for comparison
merged_stocks_df['MACD_prev'] = merged_stocks_df.groupby('ticker')['MACD'].shift(1)
merged_stocks_df['MACD_Signal_prev'] = merged_stocks_df.groupby('ticker')['MACD_Signal'].shift(1)

# Buy signal criteria: (MACD crosses above Signal Line)
merged_stocks_df.loc[(merged_stocks_df['MACD'] > merged_stocks_df['MACD_Signal']) &
                     (merged_stocks_df['MACD_prev'] <= merged_stocks_df['MACD_Signal_prev']), 'MACD_Action'] = 'Buy'

# Sell signal criteria: (MACD crosses below Signal Line)
merged_stocks_df.loc[(merged_stocks_df['MACD'] < merged_stocks_df['MACD_Signal']) &
                     (merged_stocks_df['MACD_prev'] >= merged_stocks_df['MACD_Signal_prev']), 'MACD_Action'] = 'Sell'

# Drop temporary columns
merged_stocks_df.drop(columns=['EMA_12', 'EMA_26', 'MACD_prev', 'MACD_Signal_prev'], inplace=True)

print("MACD and MACD Signals calculated and actions assigned.")
print(merged_stocks_df[['date', 'ticker', 'close', 'MACD', 'MACD_Signal', 'MACD_Action']].head(10))


MACD and MACD Signals calculated and actions assigned.
         date ticker      close      MACD  MACD_Signal MACD_Action
0  1999-11-18      A  31.473534  0.000000     0.000000        Hold
1  1999-11-19      A  28.880543 -0.206848    -0.041370        Sell
2  1999-11-22      A  31.473534 -0.159703    -0.065036        Hold
3  1999-11-23      A  28.612303 -0.349193    -0.121868        Hold
4  1999-11-24      A  29.372318 -0.433046    -0.184103        Hold
5  1999-11-26      A  29.461731 -0.486675    -0.244618        Hold
6  1999-11-29      A  30.132332 -0.469651    -0.289624        Hold
7  1999-11-30      A  30.177038 -0.447394    -0.321178        Hold
8  1999-12-01      A  30.713520 -0.382062    -0.333355        Hold
9  1999-12-02      A  31.562946 -0.258761    -0.318436         Buy


1.2 Relative Strength Index (RSI)

In [2]:
# --- Calculate RSI ---
# Function to calculate Relative Strength (RS) and then RSI
def calculate_rsi(df, period=14, column='close'):
    delta = df.groupby('ticker')[column].diff(1)
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.groupby(df['ticker']).transform(lambda x: x.ewm(span=period, adjust=False).mean())
    avg_loss = loss.groupby(df['ticker']).transform(lambda x: x.ewm(span=period, adjust=False).mean())

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

merged_stocks_df['RSI'] = calculate_rsi(merged_stocks_df, period=14)

# --- Generate RSI Signals ---
# Initialize RSI_Action column
merged_stocks_df['RSI_Action'] = 'Hold'

# Buy Signal: RSI below 30
merged_stocks_df.loc[merged_stocks_df['RSI'] < 30, 'RSI_Action'] = 'Buy'

# Sell Signal: RSI above 70
merged_stocks_df.loc[merged_stocks_df['RSI'] > 70, 'RSI_Action'] = 'Sell'

print("RSI and RSI Signals calculated and actions assigned.")
print(merged_stocks_df[['date', 'ticker', 'close', 'RSI', 'RSI_Action']].head(10))

RSI and RSI Signals calculated and actions assigned.
         date ticker      close        RSI RSI_Action
0  1999-11-18      A  31.473534        NaN       Hold
1  1999-11-19      A  28.880543   0.000000        Buy
2  1999-11-22      A  31.473534  53.571429       Hold
3  1999-11-23      A  28.612303  31.848391       Hold
4  1999-11-24      A  29.372318  39.382051       Hold
5  1999-11-26      A  29.461731  40.278215       Hold
6  1999-11-29      A  30.132332  47.052236       Hold
7  1999-11-30      A  30.177038  47.510211       Hold
8  1999-12-01      A  30.713520  53.124256       Hold
9  1999-12-02      A  31.562946  60.786489       Hold


-- Combining MACD and RSI Signals

In [3]:
# --- Combine MACD and RSI Signals ---
# The final signal is 'Buy' if both are 'Buy', 'Sell' if both are 'Sell', otherwise 'Hold'.
merged_stocks_df['Final_Signal'] = 'Hold'

merged_stocks_df.loc[(merged_stocks_df['MACD_Action'] == 'Buy') & (merged_stocks_df['RSI_Action'] == 'Buy'), 'Final_Signal'] = 'Buy'
merged_stocks_df.loc[(merged_stocks_df['MACD_Action'] == 'Sell') & (merged_stocks_df['RSI_Action'] == 'Sell'), 'Final_Signal'] = 'Sell'

print("Combined Final Signals generated.")
print(merged_stocks_df[['date', 'ticker', 'close', 'MACD_Action', 'RSI_Action', 'Final_Signal']].head(10))

# Drop rows with NaN values that might have been introduced by indicator calculations
merged_stocks_df.dropna(inplace=True)
print(f"DataFrame size after dropping NaNs from indicator calculations: {len(merged_stocks_df)}")


Combined Final Signals generated.
         date ticker      close MACD_Action RSI_Action Final_Signal
0  1999-11-18      A  31.473534        Hold       Hold         Hold
1  1999-11-19      A  28.880543        Sell        Buy         Hold
2  1999-11-22      A  31.473534        Hold       Hold         Hold
3  1999-11-23      A  28.612303        Hold       Hold         Hold
4  1999-11-24      A  29.372318        Hold       Hold         Hold
5  1999-11-26      A  29.461731        Hold       Hold         Hold
6  1999-11-29      A  30.132332        Hold       Hold         Hold
7  1999-11-30      A  30.177038        Hold       Hold         Hold
8  1999-12-01      A  30.713520        Hold       Hold         Hold
9  1999-12-02      A  31.562946         Buy       Hold         Hold
DataFrame size after dropping NaNs from indicator calculations: 18418180


2. Data Preparation and Splitting

I will integrate the engineered features and signals into our dataset. The Final_Signal will serve as our target variable. We will then split the data into training and testing sets, ensuring a chronological split to maintain the time-series integrity.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import gc

# Encode the 'Final_Signal' into numerical labels (0: Hold, 1: Buy, 2: Sell)
label_encoder = LabelEncoder()
merged_stocks_df['Signal_Encoded'] = label_encoder.fit_transform(merged_stocks_df['Final_Signal'])

# Define features (X) and target (y)
# Features will include numerical stock data, engineered indicators, and encoded categorical data.
# Exclude original 'close' (if predicting price), 'date', 'ticker', 'name', and the raw signal strings.

feature_columns = [
    'open', 'high', 'low', 'close', 'adj_close', 'volume', # Original price data
    'sector', 'industry', 'decade', # Encoded categorical and time features
    'daily_return', 'SMA_10', 'SMA_50', 'volatility_10', # Engineered features
    'MACD', 'MACD_Signal', 'RSI' # Technical indicators
]

# Ensure all feature_columns exist in the DataFrame
existing_feature_columns = [col for col in feature_columns if col in merged_stocks_df.columns]
if len(existing_feature_columns) != len(feature_columns):
    missing = set(feature_columns) - set(existing_feature_columns)
    print(f"Warning: Some feature columns are missing: {missing}. Using only existing ones.")

# --- Encode categorical columns with factorize instead of one-hot encoding ---
# factorize maps each unique value to an integer code (0..n-1).
# This avoids the huge memory cost of one-hot encoding on high-cardinality columns
# such as 'sector' and 'industry'.
for col in ['sector', 'industry']:
    if col in existing_feature_columns:
        merged_stocks_df[col] = pd.factorize(merged_stocks_df[col])[0].astype(np.int16)

X = merged_stocks_df[existing_feature_columns]
y = merged_stocks_df['Signal_Encoded']

# Perform a chronological split (e.g., 80% train, 20% test)
split_point = int(len(merged_stocks_df) * 0.8)

X_train = X.iloc[:split_point]
X_test = X.iloc[split_point:]
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]

print(f"Total data points: {len(X)}")
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print("\nClasses in target variable:")
print(y.value_counts())

print("Features (X_train head):")
print(X_train.head())
print("Target (y_train head):")
print(y_train.head())

Total data points: 18418180
Training set size: 14734544
Testing set size: 3683636

Classes in target variable:
Signal_Encoded
1    18405602
2        6338
0        6240
Name: count, dtype: int64
Features (X_train head):
        open       high        low      close  adj_close    volume  sector  \
1  30.713520  30.758226  28.478184  28.880543  25.229753  15234100       0   
2  29.551144  31.473534  28.657009  31.473534  27.494957   6577800       0   
3  30.400572  31.205294  28.612303  28.612303  24.995413   5975600       0   
4  28.701717  29.998211  28.612303  29.372318  25.659359   4843200       0   
5  29.238197  29.685265  29.148785  29.461731  25.737471   1729400       0   

   industry      MACD  MACD_Signal        RSI  
1         0 -0.206848    -0.041370   0.000000  
2         0 -0.159703    -0.065036  53.571429  
3         0 -0.349193    -0.121868  31.848391  
4         0 -0.433046    -0.184103  39.382051  
5         0 -0.486675    -0.244618  40.278215  
Target (y_train head):
1

3. Model Building and Validation

I will implement three classification models: Logistic Regression, Random Forests, and Support Vector Machines (SVM). These models will be trained on the training set, and their initial performance will be validated

In [ ]:
import numpy as np
import gc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# --- Sample a small subset of the large training set ---
rng = np.random.RandomState(42)
n = min(200_000, len(X_train))
idx = rng.choice(len(X_train), n, replace=False)
X_tr = X_train.iloc[idx] if hasattr(X_train, 'iloc') else X_train[idx]
y_tr = y_train.iloc[idx] if hasattr(y_train, 'iloc') else y_train[idx]
print(f"Training on a subset: X={X_tr.shape}, y={y_tr.shape}")

# --- Models to train ---
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs'),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "SVM":                 SVC(kernel='linear', random_state=42, C=0.1),
}

# --- Train, predict, evaluate ---
for name, model in models.items():
    print(f"\n--- Training {name} ---")
    # SVM is too slow on 200k rows -> use a smaller subset
    if name == "SVM":
        sub = rng.choice(len(X_tr), min(50_000, len(X_tr)), replace=False)
        X_fit = X_tr.iloc[sub] if hasattr(X_tr, 'iloc') else X_tr[sub]
        y_fit = y_tr.iloc[sub] if hasattr(y_tr, 'iloc') else y_tr[sub]
    else:
        X_fit, y_fit = X_tr, y_tr

    model.fit(X_fit, y_fit)
    y_pred = model.predict(X_test)
    print(f"{name} Accuracy:", accuracy_score(y_test, y_pred))
    print(f"{name} Classification Report:\n", classification_report(y_test, y_pred))
    del model; gc.collect()

Training on a subset: X=(200000, 11), y=(200000,)

--- Training Logistic Regression ---
Logistic Regression Accuracy: 0.9993150789057333


c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00      1257
           1       1.00      1.00      1.00   3681113
           2       0.00      0.00      0.00      1266

    accuracy                           1.00   3683636
   macro avg       0.33      0.33      0.33   3683636
weighted avg       1.00      1.00      1.00   3683636


--- Training Random Forest ---
Random Forest Accuracy: 0.9993150789057333


c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00      1257
           1       1.00      1.00      1.00   3681113
           2       0.00      0.00      0.00      1266

    accuracy                           1.00   3683636
   macro avg       0.33      0.33      0.33   3683636
weighted avg       1.00      1.00      1.00   3683636


--- Training SVM ---


4. Model Building and Validation

After initial training, i need to rigorously evaluate our models on the test set. This involves calculating various metrics and, if necessary, optimizing hyperparameters using techniques like cross-validation to improve performance and generalization.

In [5]:
import numpy as np
import gc
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Reuse the already-sampled X_tr / y_tr from earlier (200k subset).
# Fallback: sample here if they don't exist.
rng = np.random.RandomState(42)
if 'X_tr' not in dir():
    n = min(200_000, len(X_train))
    idx = rng.choice(len(X_train), n, replace=False)
    X_tr = X_train.iloc[idx] if hasattr(X_train, 'iloc') else X_train[idx]
    y_tr = y_train.iloc[idx] if hasattr(y_train, 'iloc') else y_train[idx]

# --- Hyperparameter Tuning for Random Forest (on a small subset) ---
print("\n--- Hyperparameter Tuning for Random Forest ---")
param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
}

# GridSearch on 50k rows only (already small enough)
n_gs = min(50_000, len(X_tr))
gs_idx = rng.choice(len(X_tr), n_gs, replace=False)
X_train_gs = X_tr.iloc[gs_idx] if hasattr(X_tr, 'iloc') else X_tr[gs_idx]
y_train_gs = y_tr.iloc[gs_idx] if hasattr(y_tr, 'iloc') else y_tr[gs_idx]
print(f"Grid Search on subset: X={X_train_gs.shape}")

grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf, cv=3, verbose=1, scoring='accuracy', n_jobs=1
)
grid_search_rf.fit(X_train_gs, y_train_gs)
print("Best parameters:", grid_search_rf.best_params_)
print("Best score:", grid_search_rf.best_score_)

# Evaluate the best model on the full test set in chunks (avoid MemoryError)
def predict_in_chunks(model, X, max_bytes=1 << 30, min_batch=10_000):
    if hasattr(X, 'values'):
        X = X.values
    n_samples, n_features = X.shape
    itemsize = X.dtype.itemsize
    batch_size = max(min_batch, int(max_bytes // (n_features * itemsize)))
    preds = []
    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        preds.append(model.predict(X[start:end]))
        gc.collect()
    return np.concatenate(preds)

best_rf_model = grid_search_rf.best_estimator_
y_pred_best_rf = predict_in_chunks(best_rf_model, X_test)
print("Optimized RF Accuracy on Test Set:", accuracy_score(y_test, y_pred_best_rf))
print("Optimized RF Classification Report:\n", classification_report(y_test, y_pred_best_rf))
del best_rf_model, grid_search_rf; gc.collect()

# --- Cross-Validation for Logistic Regression (on the small subset, lbfgs multiclass) ---
print("\n--- Cross-Validation for Logistic Regression ---")
cv_scores_lr = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs'),
    X_tr, y_tr, cv=5, scoring='accuracy', n_jobs=-1
)
print("Logistic Regression CV Scores:", cv_scores_lr)
print("Mean CV Accuracy:", np.mean(cv_scores_lr))
del cv_scores_lr; gc.collect()


--- Hyperparameter Tuning for Random Forest ---
Grid Search on subset: X=(50000, 11)
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'max_depth': 5, 'n_estimators': 50}
Best score: 0.9995000003996081


c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Optimized RF Accuracy on Test Set: 0.9993150789057333


c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

Optimized RF Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00      1257
           1       1.00      1.00      1.00   3681113
           2       0.00      0.00      0.00      1266

    accuracy                           1.00   3683636
   macro avg       0.33      0.33      0.33   3683636
weighted avg       1.00      1.00      1.00   3683636


--- Cross-Validation for Logistic Regression ---
Logistic Regression CV Scores: [0.999375 0.9994   0.99935  0.999375 0.999375]
Mean CV Accuracy: 0.999375


32